In [1]:
import argparse
import logging
import os, sys
import torch
import numpy as np
import random
import json

# To set deterministic behaviour:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')

from mmengine.config import Config, DictAction
from mmengine.logging import print_log
from mmengine.registry import RUNNERS
from mmengine.runner import Runner
from mmdet.evaluation import DumpDetResults

from mmdet.utils import setup_cache_size_limit_of_dynamo

import logging

workdir = '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Special'
os.makedirs(workdir, exist_ok=True)

def set_logger(workdir):
    """
    Sets up a logger that logs exclusively to a file.

    Parameters:
    -----------
    workdir : str
        The directory where the log file ('executor.log') will be saved.

    Returns:
    --------
    logging.Logger
        Configured logger that writes logs to a file.
    """
    # Set up the logger
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.DEBUG)

    # Remove any existing handlers
    logger.handlers = []

    # Create a file handler that logs to 'executor.log'
    file_handler = logging.FileHandler(f'{workdir}/MyNetwork.log')
    file_handler.setLevel(logging.DEBUG)

    # Create a formatter and set it for the handler
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)

    # Add the file handler to the logger
    logger.addHandler(file_handler)

    return logger


def init_cfg():
    base_folder = '/Data_large/marine/PythonProjects/MMDET/MyConfigs'
    cfg = Config.fromfile(f'{base_folder}/Sentinel_b2/vfnet_r18.py')
    return cfg


def set_seed(seed):
    # Set the seed for generating random numbers in PyTorch
    torch.manual_seed(seed)
    # If using GPUs, ensure that the random numbers are generated the same way
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    
    # Set the seed for generating random numbers in Python
    random.seed(seed)
    
    # Set the seed for generating random numbers in numpy
    np.random.seed(seed)
    
    # Ensure deterministic behavior by setting the flag
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Optionally, set environment variables to ensure reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)

### Custom

In [2]:
SEED = 71 #42 71 18 53 89
setup_cache_size_limit_of_dynamo()

# Deterministic Behaviour setting:
set_seed(SEED)
cfg = init_cfg()
cfg.randomness = dict(
    seed = SEED,
    diff_rank_seed=True,
    # deterministic=True
)


band = [5] # Selecting the Band for Venus
resize = 2048
BS = 2
LR = 0.001
random_crop = None
SENSOR = 'VENUS'
special = True

MAX_EPOCHS = 30
BAND_SEL = band # Selecting the Band list for Sentinel-2
AMP = False
selOpt = 'SGD'

logger = set_logger(workdir)



/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config Training:

In [9]:
# Normalization:
indexCorrect = {2:0, 3:1, 4:2, 8:3} if SENSOR == 'SENTINEL' else {i:i-1 for i in range(1,13,1)}
# Normalization:
MEANS=[200,154,92,63] if SENSOR == 'SENTINEL' else [158.69588,124.42161,109.27108,105.380424,88.40926,98.93067,88.819916,94.20678,103.540764,111.64337,122.92817,79.31501] 
STD=[22,24,22,60] if SENSOR == 'SENTINEL' else [34.95446,46.282494,56.252197,55.741932,64.54027,59.59095,69.65824,68.40028,77.930405,103.4634,105.30468,65.8369]
MEAN_VALS = [MEANS[indexCorrect[x]] for x in BAND_SEL]
STD_VALS = [STD[indexCorrect[x]] for x in BAND_SEL]
# Resizing:
IMG_SIZE = resize

# Annotations:
base_annot = '/Data_large/marine/Datasets/VDS2Raw/annotations' if SENSOR == 'SENTINEL' else '/Data_large/marine/Datasets/VENuS/annotations/perfect'

special_case = '_' if not special else 'special_'

ann_file = {'Train': f'{base_annot}/train_{special_case}band_{BAND_SEL[0]}.json',
            'Val': f'{base_annot}/val_{special_case}band_{BAND_SEL[0]}.json',
            'Test': f'{base_annot}/test_{special_case}band_{BAND_SEL[0]}.json',
            }

## Dataloading Directories:
data_root = '/Data_large/marine/Datasets/VDS2Raw/' if SENSOR == 'SENTINEL' else '/Data_large/marine/Datasets/VENuS' # where the images are stored
data_prefix = f'imgs/' if SENSOR == 'SENTINEL' else f'ds_L0/perfect/'


optimizers =  {'SGD':{'type': 'OptimWrapper', 'optimizer': {'type': 'SGD', 'lr': LR, 'momentum': 0.9, 'weight_decay': 0.0001}},
            'Adam':{'type': 'OptimWrapper', 'optimizer': {'type': 'Adam', 'lr': LR, 'weight_decay': 0.0001}},
            'AdamW':{'type': 'OptimWrapper', 'optimizer': {'type': 'Adam', 'lr': LR, 'weight_decay': 0.0001}},}

## Savedir:
bandsNames = ''.join([f'_b{x}' for x in BAND_SEL])
singleMulti = 'Multi' if len(BAND_SEL) > 1 else 'Single'
# WORKDIR:
kMode = {'SENTINEL':'Sentinel', 'VENUS':'VENuS'}

workdir = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/{kMode[SENSOR]}/Special/BS_{BS}/LR_{LR}/IMG_{IMG_SIZE}/BANDS_{bandsNames}/{SEED}_Optim_{selOpt}'
cfg.work_dir = workdir


# enable automatic-mixed-precision training
if AMP is True:
    optim_wrapper = cfg.optim_wrapper.type
    if optim_wrapper == 'AmpOptimWrapper':
        print_log(
            'AMP training is already enabled in your config.',
            logger='current',
            level=logging.WARNING)
    else:
        assert optim_wrapper == 'OptimWrapper', (
            '`--amp` is only supported when the optimizer wrapper type is '
            f'`OptimWrapper` but got {optim_wrapper}.')
        cfg.optim_wrapper.type = 'AmpOptimWrapper'
        cfg.optim_wrapper.loss_scale = 'dynamic'

# Dataloader:
cfg.model.data_preprocessor = dict(
    mean=[float(x) for x in MEAN_VALS],
    pad_size_divisor=1,
    std=[float(x) for x in STD_VALS],
    type='MyPrePro')

# Model Inputs:
cfg.model.backbone.in_channels = len(BAND_SEL)

# Annotation file:
cfg.train_dataloader.dataset.ann_file = ann_file['Train']
cfg.train_dataloader.dataset.data_prefix = {'img':data_prefix}
cfg.train_dataloader.dataset.data_root = data_root

cfg.val_dataloader.dataset.ann_file = ann_file['Val']
cfg.val_dataloader.dataset.data_prefix = {'img':data_prefix}
cfg.val_dataloader.dataset.data_root = data_root

#       Evaluators:
cfg.val_evaluator = dict(
    ann_file=ann_file['Val'],
    backend_args=None,
    format_only=False,
    metric='bbox',
    type='CocoMetric')

# Pipeline:
# Hook for custom loader:
loadCorrect = {2:1, 3:2, 4:3, 8:4} if SENSOR == 'SENTINEL' else {i:i for i in range(1, 13, 1)}
BAND_SEL_LOAD = [loadCorrect[x] for x in BAND_SEL]

cfg.train_dataloader.dataset.pipeline[0] = {'type': 'SelBandLoader', 'to_float32': True, 'bands_list': BAND_SEL_LOAD}
cfg.val_dataloader.dataset.pipeline[0] = {'type': 'SelBandLoader', 'to_float32': True, 'bands_list': BAND_SEL_LOAD}

cfg.train_dataloader.dataset.pipeline[3] = {'type': 'Resize', 'scale': (IMG_SIZE, IMG_SIZE), 'keep_ratio': False}
cfg.val_dataloader.dataset.pipeline[2] = {'type': 'Resize', 'scale': (IMG_SIZE, IMG_SIZE), 'keep_ratio': False}

# Metainfo
classes = ('vessel',) if SENSOR == 'SENTINEL' else ('Vessel',)
cfg.train_dataloader.dataset.metainfo = {'classes': classes, 'palette': [(220, 20, 60)]}
cfg.val_dataloader.dataset.metainfo = {'classes': classes, 'palette': [(220, 20, 60)]}


# Adding random crop to the pipeline. TODO: training with decreasing size
if random_crop is not None:
    assert isinstance(random_crop, int), 'RandomCrop Error: single dimension must be specified. E.g. 224'
    # insert random crop: 
    rc = dict(type='RandomCrop', crop_size=(random_crop, random_crop))
    cfg.train_dataloader.dataset.pipeline.insert(4, rc)
    cfg.val_dataloader.dataset.pipeline.insert(2, rc)
    

# Training params:
cfg.train_dataloader.batch_size = BS
cfg.train_cfg = {'type': 'EpochBasedTrainLoop', 'max_epochs': MAX_EPOCHS, 'val_interval': 1}

# TODO: implement stages as in: https://github.com/open-mmlab/mmdetection/blob/cfd5d3a985b0249de009b67d04f37263e11cdf3d/configs/rtmdet/rtmdet_x_p6_4xb8-300e_coco.py#L78
# lr_config = dict(policy='poly', power=0.9, min_lr=1e-4, by_epoch=False)

cfg.optim_wrapper = optimizers[selOpt]

# TODO: reset correct scheduler
cfg.param_scheduler = [{'type': 'LinearLR',
                        'start_factor': 0.001,
                        'by_epoch': True,
                        'begin': 0,
                        'convert_to_iter_based': True,
                        'end': MAX_EPOCHS//5},
                        {'type': 'MultiStepLR',
                        'begin': 0,
                        'end': MAX_EPOCHS//5,
                        'by_epoch': True,
                        'milestones': [MAX_EPOCHS//4, MAX_EPOCHS//3, MAX_EPOCHS//2],
                        'gamma': 0.75}, 
                        {# use cosine lr scheduler
                        'type':'CosineAnnealingLR',
                        'eta_min':LR * 0.05,
                        'begin':MAX_EPOCHS//2,
                        'end':MAX_EPOCHS,
                        'T_max':MAX_EPOCHS//1.5,
                        'by_epoch':True,
                        'convert_to_iter_based':True,}
                        ]

#### Test Config hooks:
default_hooks = cfg.default_hooks
if 'visualization' in default_hooks:
    visualization_hook = default_hooks['visualization']
    # Turn on visualization
    visualization_hook['draw'] = False

cfg.test_dataloader = dict(
            batch_size=1,
            dataset=dict(
                ann_file=ann_file['Test'],
                data_root=data_root,
                data_prefix=dict(img=data_prefix),
                filter_cfg=dict(filter_empty_gt=True),
                metainfo=dict(classes=classes, palette=[
                    (
                        220,
                        20,
                        60,
                    ),
                ]),
                pipeline=[{'type': 'SelBandLoader', 'to_float32': True, 'bands_list': BAND_SEL_LOAD},
                    dict(type='LoadAnnotations', with_bbox=True),
                    dict(keep_ratio=False, scale=(IMG_SIZE,IMG_SIZE,), type='Resize'),
                    dict(
                        meta_keys=('img_path', 'img_id', 'seg_map_path', 
                                'height', 'width', 'instances', 'sample_idx', 
                                'img', 'img_shape', 'ori_shape', 'scale', 'scale_factor', 
                                'keep_ratio', 'homography_matrix', 'gt_bboxes', 'gt_ignore_flags', 
                                'gt_bboxes_labels'),
                        type='PackDetInputs'),
                ],
                test_mode=True,
                type='CocoDataset'),
            drop_last=False,
            num_workers=2,
            persistent_workers=True,
            sampler=dict(shuffle=False, type='DefaultSampler'))

cfg.test_evaluator = dict(
            type='CocoMetric',
            metric='bbox',
            format_only=False,
            ann_file=ann_file['Test'],
            outfile_prefix=f'{workdir}/test_results')

### Model Customization:

In [ ]:
cfg.model.backbone = {'type': 'TimmEncoder',
                    'model_name': 'resnet-18', 
                    'features_only': True, 
                    'pretrained': True, 
                    'in_chans': 1, 
                    'frozen_stages':1,
                    }

#### Backtesting:

In [ ]:
import timm

m = timm.create_model('resnest26d', features_only=True, pretrained=True, in_chans=10)
o = m(torch.randn(2, 10, 1024, 1024))
o = tuple(o)

for x in o:
    print(x.shape)
    
 
in_channels_neck = []
for item in out:
    in_channels_neck.append(item.shape[1])
    print(item.shape)

In [ ]:
cfg.model.neck = {'type': 'FPN',
                    'in_channels': in_channels_neck,
                    'out_channels': 256,
                    'num_outs': len(in_channels_neck),
                    'start_level': 0,
                    'end_level': 3,
                    'add_extra_convs': False,
                    'relu_before_extra_convs': True,
                    'no_norm_on_lateral': True,
                    'conv_cfg': None,
                    'norm_cfg': None,
                    'act_cfg': None,
                    'upsample_cfg': dict(mode='nearest'),
                    'init_cfg': dict(type='Xavier', layer='Conv2d', distribution='uniform')
                    }

In [ ]:
cfg.model.head 

## RUNNER SET

Setting the runner from cfg 

In [10]:
# build the runner from config
if 'runner_type' not in cfg:
    # build the default runner
    runner = Runner.from_cfg(cfg)
else:
    # build customized runner from the registry
    # if 'runner_type' is set in the cfg
    runner = RUNNERS.build(cfg)

08/26 16:19:53 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.19 | packaged by conda-forge | (default, Mar 20 2024, 12:47:35) [GCC 12.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 71
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda-11.4
    NVCC: Cuda compilation tools, release 11.4, V11.4.152
    GCC: gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


FileNotFoundError: [Errno 2] No such file or directory: '/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Special/BS_2/LR_0.001/IMG_2048/BANDS__b5/Optim_SGD/20240826_144212/vis_data/config.py'

#### RUNNER START

In [5]:
runner.train()

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
08/26 14:42:23 - mmengine - INFO - load model from: torchvision://resnet18
08/26 14:42:23 - mmengine - INFO - Loads checkpoint by torchvision backend from path: torchvision://resnet18
08/26 14:42:23 - mmengine - WARNING - The model and loaded state dict do not match exactly

size mismatch for conv1.weight: copying a param with shape torch.Size([64, 3, 7, 7]) from checkpoint, the shape in current model is torch.Size([64, 1, 7, 7]).
unexpected key in source state_dict: fc.weight, fc.bias

08/26 14:42:25 - mmengine - WARNING - "FileClient" will be deprecated in future. Please use io functions in https://mmengine.readthedocs.io/en/latest/api/fileio.html#file-io
08/26 14:42:25 - mmengine - WARNING - "HardDiskBackend" is the alias of "LocalBackend

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/26 14:42:43 - mmengine - INFO - Epoch(train)  [1][ 1/70]  lr: 1.0000e-06  eta: 11:00:41  time: 18.8858  data_time: 4.7555  memory: 5297  loss: 1.0005  loss_cls: 0.0137  loss_bbox: 0.3847  loss_bbox_rf: 0.6021
08/26 14:42:45 - mmengine - INFO - Epoch(train)  [1][ 2/70]  lr: 3.3842e-06  eta: 6:03:57  time: 10.4089  data_time: 2.3888  memory: 5374  loss: 1.0831  loss_cls: 0.0129  loss_bbox: 0.4175  loss_bbox_rf: 0.6527
08/26 14:42:47 - mmengine - INFO - Epoch(train)  [1][ 3/70]  lr: 5.7685e-06  eta: 4:23:30  time: 7.5397  data_time: 1.5961  memory: 5374  loss: 0.9877  loss_cls: 0.0145  loss_bbox: 0.3777  loss_bbox_rf: 0.5956
08/26 14:42:49 - mmengine - INFO - Epoch(train)  [1][ 4/70]  lr: 8.1527e-06  eta: 3:30:18  time: 6.0202  data_time: 1.2061  memory: 5374  loss: 1.2662  loss_cls: 0.0138  loss_bbox: 0.4840  loss_bbox_rf: 0.7684
08/26 14:42:50 - mmengine - INFO - Epoch(train)  [1][ 5/70]  lr: 1.0537e-05  eta: 2:59:00  time: 5.1266  data_time: 0.9690  memory: 5374  loss: 1.2668  loss_

VFNet(
  (data_preprocessor): MyPrePro()
  (backbone): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): ResLayer(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momen

#### RUNNER TEST START

In [6]:
runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{workdir}/test_result/test.pkl'))
# start testing
output_test_data =runner.test()

# Specify the file name
file_name = f'{workdir}/test_result/coco_metrics.json'# Specify the filepath
# Write the dictionary to a JSON file
with open(file_name, 'w') as json_file:
    json.dump(output_test_data, json_file, indent=4)

print(f"Data has been saved to {file_name}")

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
08/26 16:01:49 - mmengine - WARNING - The prefix is not set in metric class DumpDetResults.
08/26 16:01:51 - mmengine - INFO - Epoch(test) [ 1/57]    eta: 0:01:58  time: 2.1196  data_time: 1.7812  memory: 1220  
08/26 16:01:52 - mmengine - INFO - Epoch(test) [ 2/57]    eta: 0:01:08  time: 1.2541  data_time: 0.9599  memory: 1220  
08/26 16:01:53 - mmengine - INFO - Epoch(test) [ 3/57]    eta: 0:01:06  time: 1.2230  data_time: 0.9123  memory: 1220  
08/26 16:01:56 - mmengine - INFO - Epoch(test) [ 4/57]    eta: 0:01:26  time: 1.6266  data_time: 1.2977  memory: 1220  
08/26 16:01:56 - mmengine - INFO - Epoch(test) [ 5/57]    eta: 0:01:11  time: 1.3795  data_time: 1.0393  memory: 1220  
08/26 16:02:14 - mmengine - INFO - Epoch(test) [ 6/57]    eta: 0:03:32  time: 4.1706  data_time: 3.8332  memory: 1220  
08/26 16:02:15 - mmeng